In [1]:
# 🎓 Student Performance & Academic Risk Analysis System
**Author:** Simeon Ogechukwu   
**Role:** Aspiring Educational Technology Systems Manager  
**Dataset:** Real-World Portuguese Secondary Student Performance Dataset (Cortez & Silva, 2008)

---

### 📌 Project Executive Summary
This project analyzes academic, demographic, and behavioral factors affecting secondary school student performance. 
Using automated logic, it:
1. Standardizes raw grades to a 0–100% scale.
2. Flags students requiring immediate academic intervention based on attendance thresholds and exam performance.
3. Evaluates the impact of home internet access and study habits on final outcomes.
4. Generates an automated export of at-risk students for academic counseling teams.

SyntaxError: invalid character '–' (U+2013) (3286283728.py, line 11)

# ==========================================================
# STEP 1: IMPORT LIBRARIES & LOAD REAL-WORLD DATASET
# ==========================================================

# pandas: for working with data tables (DataFrames)
import pandas as pd

# matplotlib.pyplot: for creating visual charts
import matplotlib.pyplot as plt

# seaborn: makes charts look cleaner and more modern
import seaborn as sns

# Set the background grid style for all future charts
sns.set_style("whitegrid")

# The web link (URL) directly pointing to the real student dataset on GitHub
# This dataset contains records from Portuguese secondary school students
url = "https://raw.githubusercontent.com/guipsamora/pandas_exercises/master/04_Apply/Students_Alcohol_Consumption/student-mat.csv"

# pd.read_csv() downloads and reads the CSV data directly from the internet
df = pd.read_csv(url)

# Display confirmation and dataset dimensions
print("✅ REAL DATASET LOADED SUCCESSFULLY!")
print(f"Total students in dataset: {len(df)}")        # Number of rows
print(f"Total variables collected: {len(df.columns)}") # Number of columns

# Show the first 5 rows, selecting only the most interesting columns to keep it readable
print("\nFirst 5 student records:")
df[['sex', 'age', 'studytime', 'failures', 'internet', 'absences', 'G1', 'G2', 'G3']].head()

In [ ]:
# ==========================================================
# STEP 2: RESCALING GRADES & BASELINE STATISTICS
# ==========================================================

# In Portugal, grades are scored from 0 to 20 points (10 is the passing mark).
# To make this easier for anyone reading our report, we convert it to a 0-100% scale.
# Formula: (Grade / 20) * 100

# G3 is the final exam score
df['final_score'] = (df['G3'] / 20) * 100

# G1 is the first-period (midterm) score
df['midterm_score'] = (df['G1'] / 20) * 100

# .describe() computes count, mean, std, min, 25%, 50%, 75%, and max automatically
print("--- SUMMARY STATISTICS ---")
print(df[['age', 'absences', 'studytime', 'final_score']].describe())

# Calculate overall school-wide averages using .mean()
avg_final = df['final_score'].mean()
avg_absences = df['absences'].mean()

# Print the formatted results (: .1f limits the output to 1 decimal place)
print(f"\nAverage Final Score across all students: {avg_final:.1f}%")
print(f"Average Days Absent: {avg_absences:.1f} days")



In [ ]:
# ==========================================================
# STEP 3: BUSINESS LOGIC (FUNCTIONS & FEATURE ENGINEERING)
# ==========================================================

# FUNCTION 1: Convert numeric percentage into standard letter grades (A, B, C, D, F)
def assign_grade(score):
    """Takes a percentage score and returns a letter grade string"""
    if score >= 80:
        return 'A'
    elif score >= 65:
        return 'B'
    elif score >= 50:
        return 'C'
    elif score >= 40:
        return 'D'
    else:
        return 'F (Fail)'

# FUNCTION 2: Flag students who need academic help
# An EdTech rule: If a student missed 10+ days OR scored below 50%, flag them 'At-Risk'
def identify_risk(row):
    """
    Evaluates an entire student row.
    Looks at 'absences' and 'final_score' together.
    """
    if row['absences'] >= 10 or row['final_score'] < 50:
        return 'At-Risk'
    else:
        return 'Good Standing'

# .apply(assign_grade) runs the grading function on every number in the 'final_score' column
df['letter_grade'] = df['final_score'].apply(assign_grade)

# When using multiple columns inside a function, we use .apply(..., axis=1)
# axis=1 tells pandas: "Pass the ENTIRE row (all columns) into the function"
df['risk_status'] = df.apply(identify_risk, axis=1)

# Display top 10 rows showing our newly calculated columns
print("--- DATA WITH NEW GRADES & RISK STATUS ---")
print(df[['sex', 'age', 'absences', 'final_score', 'letter_grade', 'risk_status']].head(10))

# Count how many total students are in Good Standing vs At-Risk
print("\n--- RISK STATUS BREAKDOWN ---")
print(df['risk_status'].value_counts())

In [ ]:
# ==========================================================
# STEP 4: GROUP ANALYSIS (ANSWERING REAL QUESTIONS)
# ==========================================================

# In the raw data, 'studytime' is coded as numbers 1, 2, 3, 4.
# Let's map these numbers into readable text descriptions:
# 1 = less than 2 hours, 2 = 2 to 5 hours, etc.
studytime_mapping = {
    1: '< 2 hours', 
    2: '2 - 5 hours', 
    3: '5 - 10 hours', 
    4: '> 10 hours'
}

# .map() replaces the numbers with our descriptive labels
df['studytime_label'] = df['studytime'].map(studytime_mapping)

# QUESTION 1: Does studying more hours actually lead to higher grades?
# Group by study time, calculate both the student count and average final score
study_impact = df.groupby('studytime_label')['final_score'].agg(['count', 'mean']).sort_values('mean', ascending=False)

print("--- FINAL SCORE BY WEEKLY STUDY TIME ---")
print(study_impact)


# QUESTION 2: Do students with home internet get higher scores than those without?
# Group by 'internet' column ('yes' vs 'no') and calculate averages
internet_impact = df.groupby('internet')['final_score'].agg(['count', 'mean'])

print("\n--- FINAL SCORE BY INTERNET ACCESS ---")
print(internet_impact)

In [ ]:
# ==========================================================
# STEP 5: VISUAL DASHBOARD (4 CHARTS IN 1 FIGURE)
# ==========================================================

# plt.subplots(2, 2) creates a grid of 2 rows and 2 columns (4 chart spaces)
# figsize=(14, 10) sets width to 14 inches and height to 10 inches
fig, axes = plt.subplots(2, 2, figsize=(14, 10))


# --- CHART 1: Top-Left (axes[0, 0]) -> Bar Chart of Letter Grades ---
# Specify order so bars display A -> B -> C -> D -> F
grade_order = ['A', 'B', 'C', 'D', 'F (Fail)']
grade_counts = df['letter_grade'].value_counts().reindex(grade_order)

grade_counts.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Overall Grade Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Number of Students')
axes[0, 0].tick_params(axis='x', rotation=0) # Keep grade labels horizontal


# --- CHART 2: Top-Right (axes[0, 1]) -> Bar Chart of Risk Status ---
risk_counts = df['risk_status'].value_counts()

# Use green for Good Standing and red/crimson for At-Risk
risk_counts.plot(kind='bar', ax=axes[0, 1], color=['green', 'crimson'])
axes[0, 1].set_title('At-Risk vs Good Standing Breakdown', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Number of Students')
axes[0, 1].tick_params(axis='x', rotation=0)


# --- CHART 3: Bottom-Left (axes[1, 0]) -> Bar Chart of Study Time vs Score ---
study_impact['mean'].plot(kind='bar', ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Average Final Score by Study Time', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Average Final Score (%)')
axes[1, 0].tick_params(axis='x', rotation=0)


# --- CHART 4: Bottom-Right (axes[1, 1]) -> Scatter Plot: Absences vs Score ---
# Each dot is one student (X = their absences, Y = their score)
axes[1, 1].scatter(df['absences'], df['final_score'], alpha=0.5, color='purple')
axes[1, 1].set_title('Absences vs Final Score Relationship', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Absences')
axes[1, 1].set_ylabel('Final Score (%)')


# tight_layout automatically adjusts spacing so titles and labels don't overlap
plt.tight_layout()

# Render all charts to screen
plt.show()


In [ ]:
# ==========================================================
# STEP 6: FILTER & EXPORT AT-RISK STUDENTS TO CSV
# ==========================================================

# Filter our table: keep ONLY students where risk_status is 'At-Risk'
at_risk = df[df['risk_status'] == 'At-Risk']

# Select the key columns needed by academic advisors and sort by lowest score first
intervention_list = at_risk[['sex', 'age', 'studytime_label', 'absences', 'final_score', 'letter_grade']].sort_values('final_score')

print("--- STUDENTS REQUIRING ACADEMIC INTERVENTION ---")
print(f"Total students flagged: {len(intervention_list)} out of {len(df)}")
print("\nTop 10 highest priority students for intervention:")
print(intervention_list.head(10))

# .to_csv() exports this filtered data directly to a clean CSV spreadsheet on your Mac
# index=False prevents pandas from writing an unnecessary row-number column
intervention_list.to_csv('real_students_intervention_list.csv', index=False)

print("\n📁 File successfully exported as 'real_students_intervention_list.csv'!")

In [ ]:
# ==========================================================
# STEP 7: DAY 5 CHALLENGE SOLUTION
# ==========================================================

# Filter condition:
# 1. No internet at home: (df['internet'] == 'no')
# 2. Studied less than 2 hours: (df['studytime'] == 1)
# Combine both with the '&' (AND) operator

low_resource_students = df[(df['internet'] == 'no') & (df['studytime'] == 1)]

# Calculate count and average score
count = len(low_resource_students)
average_score = low_resource_students['final_score'].mean()

print("--- CHALLENGE RESULTS ---")
print(f"Number of students with no internet & <2hrs study time: {count}")
print(f"Their average final score: {average_score:.1f}%")
print(f"Compared to school-wide average ({avg_final:.1f}%): Difference is {avg_final - average_score:.1f}% lower")


In [ ]:
---
## 📊 Key Educational Insights & Recommendations

1. **Study Time Matters:** Students studying >10 hours/week outperformed students studying <2 hours/week by a significant margin.
2. **The Digital Divide:** Students with home internet access scored consistently higher than those without.
3. **Actionable Intervention:** An automated risk threshold (Absences >= 10 OR Score < 50%) successfully isolated vulnerable students into an exportable CSV for academic support teams.